# EAR Threshold Sensitivity Analysis

Controlled sensitivity analysis for the Sensors manuscript.

**Design:** Random Forest, T=30, Leave-One-Driver-Out (LODO), thresholds 0.19–0.23. Only the binary `eye_closed` feature changes across thresholds. The notebook first reports the 0.21 result as the reproduction checkpoint.


In [ ]:
import re, time, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)
warnings.filterwarnings('ignore')

CSV_PATH = 'eye_features.csv'
SEQ_LEN = 30
THRESHOLDS = [0.19, 0.20, 0.21, 0.22, 0.23]
RANDOM_STATE = 42

# RF hyperparameters
RF_PARAMS = dict(n_estimators=300, class_weight='balanced',
                 random_state=RANDOM_STATE, n_jobs=-1)


In [ ]:
df = pd.read_csv(CSV_PATH)
required = {'image_name','ear','label'}
assert required.issubset(df.columns), f'Missing columns: {required-set(df.columns)}'
print('Rows:', len(df))
print(df[['image_name','ear','label']].head())


In [ ]:
from pathlib import Path

# Parse driver, scenario and frame number from image_name.
# Expected pattern resembles: 001_glasses_sleepyCombination_1000_drowsy.jpg
def parse_name(name):
    stem = Path(str(name)).stem
    parts = stem.split('_')
    driver = parts[0]
    # Find the last numeric token; this is used as the frame index.
    numeric_positions = [i for i,p in enumerate(parts) if p.isdigit()]
    if not numeric_positions:
        raise ValueError(f'Cannot find frame number in {name}')
    j = numeric_positions[-1]
    frame = int(parts[j])
    scenario = '_'.join(parts[1:j])
    return driver, scenario, frame

parsed = df['image_name'].apply(parse_name)
df[['driver_id','scenario','frame_idx']] = pd.DataFrame(parsed.tolist(), index=df.index)
df['stream_id'] = df['driver_id'].astype(str) + '__' + df['scenario'].astype(str)
df = df.sort_values(['driver_id','scenario','frame_idx']).reset_index(drop=True)
print('Drivers:', sorted(df.driver_id.unique()))
print('Streams:', df.stream_id.nunique())


In [ ]:
def engineer_and_sequence(base_df, threshold, seq_len=30):
    d = base_df.copy()
    # Engineering is performed separately within each original stream.
    d['ear_rolling_mean'] = d.groupby('stream_id')['ear'].transform(lambda s: s.rolling(5, min_periods=1).mean())
    d['ear_delta'] = d.groupby('stream_id')['ear'].diff().fillna(0.0)
    d['eye_closed'] = (d['ear'] < threshold).astype(int)

    features = ['ear','ear_rolling_mean','ear_delta','eye_closed']
    Xs, ys, drivers = [], [], []
    for _, g in d.groupby('stream_id', sort=False):
        g = g.sort_values('frame_idx').reset_index(drop=True)
        if len(g) < seq_len:
            continue
        a = g[features].to_numpy(dtype=np.float32)
        labels = g['label'].to_numpy()
        drv = g['driver_id'].iloc[0]
        for end in range(seq_len-1, len(g)):
            start = end-seq_len+1
            Xs.append(a[start:end+1])
            ys.append(labels[end])
            drivers.append(drv)
    return np.asarray(Xs), np.asarray(ys), np.asarray(drivers)

def evaluate_threshold(base_df, threshold):
    X, y, drivers = engineer_and_sequence(base_df, threshold, SEQ_LEN)
    # RF receives flattened T x 4 sequences.
    X = X.reshape(len(X), -1)
    rows = []
    for held_out in sorted(np.unique(drivers)):
        train = drivers != held_out
        test = drivers == held_out

        # Fit scaler ONLY on the LODO training partition.
        scaler = MinMaxScaler()
        Xtr = scaler.fit_transform(X[train])
        Xte = scaler.transform(X[test])

        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(Xtr, y[train])
        pred = model.predict(Xte)
        prob = model.predict_proba(Xte)[:,1]

        rows.append({
            'threshold': threshold,
            'held_out_driver': held_out,
            'n_test': int(test.sum()),
            'accuracy': accuracy_score(y[test], pred),
            'precision_drowsy': precision_score(y[test], pred, pos_label=1, zero_division=0),
            'recall_drowsy': recall_score(y[test], pred, pos_label=1, zero_division=0),
            'f1_drowsy': f1_score(y[test], pred, pos_label=1, zero_division=0),
            'macro_f1': f1_score(y[test], pred, average='macro', zero_division=0),
            'weighted_f1': f1_score(y[test], pred, average='weighted', zero_division=0),
            'auroc': roc_auc_score(y[test], prob),
            'auprc': average_precision_score(y[test], prob)
        })
    return pd.DataFrame(rows)


In [ ]:
# STEP 1 — Reproduction checkpoint at EAR = 0.21
t0 = time.time()
checkpoint = evaluate_threshold(df, 0.21)
display(checkpoint.round(4))
metrics = ['accuracy','precision_drowsy','recall_drowsy','f1_drowsy','macro_f1','weighted_f1','auroc','auprc']
checkpoint_summary = pd.DataFrame({
    'mean': checkpoint[metrics].mean(),
    'sd': checkpoint[metrics].std(ddof=1)
})
print('\n0.21 reproduction checkpoint (fold mean ± SD):')
display(checkpoint_summary.round(4))
print(f'Elapsed: {(time.time()-t0)/60:.1f} minutes')
print('\nPublished approximate targets: Accuracy 0.556; F1-drowsy 0.618; AUROC 0.626; AUPRC 0.663')
print('Do not interpret the sensitivity analysis unless this checkpoint is acceptably reproduced.')


In [ ]:
# STEP 2 — Run the remaining thresholds only after checking 0.21 above.
all_folds = [checkpoint]
for th in [0.19, 0.20, 0.22, 0.23]:
    print(f'Running threshold {th:.2f} ...')
    t0 = time.time()
    r = evaluate_threshold(df, th)
    all_folds.append(r)
    print(f'Finished {th:.2f} in {(time.time()-t0)/60:.1f} minutes')

fold_results = pd.concat(all_folds, ignore_index=True)
summary_rows = []
for th, g in fold_results.groupby('threshold'):
    row = {'threshold': th}
    for m in metrics:
        row[m+'_mean'] = g[m].mean()
        row[m+'_sd'] = g[m].std(ddof=1)
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows).sort_values('threshold')

display_cols = ['threshold','accuracy_mean','accuracy_sd','f1_drowsy_mean','f1_drowsy_sd',
                'auroc_mean','auroc_sd','auprc_mean','auprc_sd']
print('\nThreshold sensitivity summary:')
display(summary[display_cols].round(4))

fold_results.to_csv('ear_threshold_sensitivity_fold_results.csv', index=False)
summary.to_csv('ear_threshold_sensitivity_summary.csv', index=False)
print('\nSaved: ear_threshold_sensitivity_fold_results.csv')
print('Saved: ear_threshold_sensitivity_summary.csv')


In [ ]:
# Sensitivity analysis: EAR threshold = 0.19 only
th = 0.19

print(f'Running threshold {th:.2f} ...')
t0 = time.time()

result_019 = evaluate_threshold(df, th)

print(f'Finished {th:.2f} in {(time.time()-t0)/60:.1f} minutes')

# Show fold-level results
display(result_019.round(4))

# Show mean ± SD
summary_019 = result_019[metrics].agg(['mean', 'std']).T
print('\nThreshold 0.19: fold mean ± SD')
display(summary_019.round(4))

# Save immediately
result_019.to_csv('ear_threshold_019_fold_results.csv', index=False)
print('Saved: ear_threshold_019_fold_results.csv')

In [ ]:
# Sensitivity analysis: EAR threshold = 0.20 only
th = 0.20

print(f'Running threshold {th:.2f} ...')
t0 = time.time()

result_020 = evaluate_threshold(df, th)

print(f'Finished {th:.2f} in {(time.time()-t0)/60:.1f} minutes')

# Show fold-level results
display(result_020.round(4))

# Show mean ± SD
summary_020 = result_020[metrics].agg(['mean', 'std']).T
print('\nThreshold 0.20: fold mean ± SD')
display(summary_020.round(4))

# Save immediately
result_020.to_csv('ear_threshold_020_fold_results.csv', index=False)
print('Saved: ear_threshold_020_fold_results.csv')

In [ ]:
# Sensitivity analysis: EAR threshold = 0.22 only
th = 0.22

print(f'Running threshold {th:.2f} ...')
t0 = time.time()

result_022 = evaluate_threshold(df, th)

print(f'Finished {th:.2f} in {(time.time()-t0)/60:.1f} minutes')

# Show fold-level results
display(result_022.round(4))

# Show mean ± SD
summary_022 = result_022[metrics].agg(['mean', 'std']).T
print('\nThreshold 0.22: fold mean ± SD')
display(summary_022.round(4))

# Save immediately
result_022.to_csv('ear_threshold_022_fold_results.csv', index=False)
print('Saved: ear_threshold_022_fold_results.csv')

In [ ]:
# Sensitivity analysis: EAR threshold = 0.23 only
th = 0.23

print(f'Running threshold {th:.2f} ...')
t0 = time.time()

result_023 = evaluate_threshold(df, th)

print(f'Finished {th:.2f} in {(time.time()-t0)/60:.1f} minutes')

# Show fold-level results
display(result_023.round(4))

# Show mean ± SD
summary_023 = result_023[metrics].agg(['mean', 'std']).T
print('\nThreshold 0.23: fold mean ± SD')
display(summary_023.round(4))

# Save immediately
result_023.to_csv('ear_threshold_023_fold_results.csv', index=False)
print('Saved: ear_threshold_023_fold_results.csv')